Code Architecture -

Discharge PDF
      ↓
RAG → Groq → Structured JSON
      ↓
Store JSON in memory
      ↓
User Question
      ↓
Groq reasons over structured summary
      ↓
Answer

# Install Gradio for Leightweight Interactive UI

In [ ]:
!pip install gradio


# Check GPU enabled?

In [ ]:
import torch
torch.cuda.is_available()


True

# Install required Libraries


In [ ]:
!pip install faiss-cpu sentence-transformers groq pypdf numpy
!pip install pypdf



# Set Groq API Key

In [ ]:
import os
from google.colab import userdata # Use userdata for API key
from groq import Groq

# Retrieve GROQ_API_KEY from Colab secrets. Ensure it's set in the Colab secrets panel.
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq client initialized successfully.")

Groq client initialized successfully.


# Discharge PDF


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving synthetic_patient_4.pdf to synthetic_patient_4 (4).pdf
Saving synthetic_patient_3.pdf to synthetic_patient_3 (4).pdf
Saving synthetic_patient_2.pdf to synthetic_patient_2 (4).pdf
Saving synthetic_patient_1.pdf to synthetic_patient_1 (4).pdf
Saving Patient Discharge Summary.pdf to Patient Discharge Summary (9).pdf


# Extract text from PDF


In [ ]:
from pypdf import PdfReader

def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


# Dictionary to store patient documents
all_documents = {}

# Loop through all uploaded PDFs
for pdf_name in uploaded.keys():

    print(f"Processing: {pdf_name}")

    document_text = extract_text(pdf_name)

    # Create clean patient_id from filename
    patient_id = pdf_name.replace(".pdf", "").replace(" ", "_")

    all_documents[patient_id] = document_text

    print(f"Loaded {patient_id} | Length: {len(document_text)} characters\n")

print("Patients loaded:", list(all_documents.keys()))

Processing: synthetic_patient_4 (4).pdf
Loaded synthetic_patient_4_(4) | Length: 1139 characters

Processing: synthetic_patient_3 (4).pdf
Loaded synthetic_patient_3_(4) | Length: 1127 characters

Processing: synthetic_patient_2 (4).pdf
Loaded synthetic_patient_2_(4) | Length: 1146 characters

Processing: synthetic_patient_1 (4).pdf
Loaded synthetic_patient_1_(4) | Length: 1079 characters

Processing: Patient Discharge Summary (9).pdf
Loaded Patient_Discharge_Summary_(9) | Length: 6610 characters

Patients loaded: ['synthetic_patient_4_(4)', 'synthetic_patient_3_(4)', 'synthetic_patient_2_(4)', 'synthetic_patient_1_(4)', 'Patient_Discharge_Summary_(9)']


# Chunking - Chunk the text

In [ ]:
# ==========================
# Multi-Patient Chunking
# ==========================

def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks


all_chunks = []
chunk_metadata = []

for patient_id, document_text in all_documents.items():

    print(f"Chunking patient: {patient_id}")

    patient_chunks = chunk_text(document_text)

    for chunk in patient_chunks:
        all_chunks.append(chunk)
        chunk_metadata.append({
            "patient_id": patient_id
        })

    print(f"  → {len(patient_chunks)} chunks created\n")

print("Total chunks across all patients:", len(all_chunks))

Chunking patient: synthetic_patient_4_(4)
  → 2 chunks created

Chunking patient: synthetic_patient_3_(4)
  → 2 chunks created

Chunking patient: synthetic_patient_2_(4)
  → 2 chunks created

Chunking patient: synthetic_patient_1_(4)
  → 2 chunks created

Chunking patient: Patient_Discharge_Summary_(9)
  → 11 chunks created

Total chunks across all patients: 19


# Embeddings (sentence-transformers)

In [ ]:
# ==========================
# Embedding Section (Multi-Patient)
# ==========================

from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings for all chunks...")
all_embeddings = embedding_model.encode(all_chunks)

all_embeddings = np.array(all_embeddings).astype("float32")

print("Embeddings shape:", all_embeddings.shape)


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for all chunks...
Embeddings shape: (19, 384)


# FAISS Vector DB


In [ ]:
# ==========================
# Build FAISS Index
# ==========================

import faiss

dimension = all_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(all_embeddings)

print("FAISS index built successfully.")
print("Total vectors indexed:", index.ntotal)


FAISS index built successfully.
Total vectors indexed: 19


# Retrieve top chunks


In [ ]:
# ==========================
# Patient-Aware Retrieval
# ==========================

def retrieve_patient_context(query, selected_patient, k=5):

    query_embedding = embedding_model.encode([query]).astype("float32")

    # Get indices for selected patient only
    patient_indices = [
        i for i, meta in enumerate(chunk_metadata)
        if meta["patient_id"] == selected_patient
    ]

    if not patient_indices:
        return [], []

    # Build temporary FAISS index for selected patient
    patient_embeddings = all_embeddings[patient_indices]

    temp_index = faiss.IndexFlatL2(all_embeddings.shape[1])
    temp_index.add(patient_embeddings)

    distances, indices = temp_index.search(
        query_embedding,
        min(k, len(patient_indices))
    )

    retrieved_chunks = []
    retrieved_chunk_ids = []

    for idx in indices[0]:
        original_index = patient_indices[idx]
        retrieved_chunks.append(all_chunks[original_index])
        retrieved_chunk_ids.append(original_index)

    return retrieved_chunks, retrieved_chunk_ids


# Structured Clinical Summary (JSON)


In [ ]:
# ==========================
# Generate Structured Summary Per Patient
# ==========================

def generate_structured_summary(patient_id):

    document_text = all_documents[patient_id]

    prompt = f"""
You are a clinical discharge extraction assistant.

Extract structured clinical information from the content below.

Return STRICT JSON with the following schema:

{{
  "patient_demographics": {{
      "name": "",
      "age": "",
      "gender": "",
      "dob": "",
      "mrn": "",
      "admission_date": "",
      "discharge_date": "",
      "length_of_stay": ""
  }},
  "admission_vitals": {{
      "blood_pressure": "",
      "heart_rate": "",
      "respiratory_rate": "",
      "temperature": "",
      "oxygen_saturation": ""
  }},
  "primary_diagnosis": "",
  "secondary_diagnoses": [],
  "procedures_performed": [],
  "medication_changes": {{
      "stopped": [],
      "started": [],
      "adjusted": []
  }},
  "lab_trend_summary": {{
      "creatinine_trend": "",
      "infection_markers_trend": "",
      "lactate_trend": ""
  }},
  "allergies": [],
  "social_factors": [],
  "follow_up_instructions": [],
  "risk_flags": [],
  "readmission_risk_indicators": []
}}

Rules:
- If data not present, leave empty.
- Do NOT hallucinate.
- Use ONLY the provided content.

Content:
{document_text}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": "You are a strict clinical extraction engine. Return ONLY valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )

    summary = response.choices[0].message.content

    return summary

# Store Clinical Summary from JSON

In [ ]:
# ==========================
# Generate & Store Structured Summary Per Patient
# ==========================

patient_summaries = {}

def generate_structured_summary(patient_id):
    """
    Generate structured clinical summary JSON
    for a specific patient.
    """

    # Retrieve full document text for selected patient
    document_text = all_documents[patient_id]

    prompt = f"""
You are a clinical discharge extraction assistant.

Extract structured clinical information from the content below.

Return STRICT JSON with the defined schema.

Rules:
- Do NOT hallucinate.
- Use ONLY the provided content.
- If data is not present, leave fields empty.
- Return JSON only.

Content:
{document_text}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": "You are a strict clinical extraction engine. Return ONLY valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0,
        response_format={"type": "json_object"}
    )

    return response.choices[0].message.content


# ==========================
# Generate summaries for all uploaded patients
# ==========================

for patient_id in all_documents.keys():
    print(f"Generating structured summary for {patient_id}...")
    patient_summaries[patient_id] = generate_structured_summary(patient_id)

print("All patient summaries generated successfully.")

Generating structured summary for synthetic_patient_4_(4)...
Generating structured summary for synthetic_patient_3_(4)...
Generating structured summary for synthetic_patient_2_(4)...
Generating structured summary for synthetic_patient_1_(4)...
Generating structured summary for Patient_Discharge_Summary_(9)...
All patient summaries generated successfully.


# Create Chat Function With Memory


In [ ]:
def chat_with_memory(message, history, selected_patients):
    """
    Handles both single-patient and multi-patient comparison mode.
    """

    # -------------------------
    # Validate patient selection
    # -------------------------
    if not selected_patients:
        return "⚠ Please select at least one patient."

    # Ensure selected_patients is always a list
    if isinstance(selected_patients, str):
        selected_patients = [selected_patients]

    # -------------------------
    # Combine structured summaries
    # -------------------------
    combined_summary = ""

    for patient in selected_patients:
        combined_summary += f"\n\n=== Patient: {patient} ===\n"
        combined_summary += str(patient_summaries[patient])

    # -------------------------
    # Retrieve relevant chunks
    # -------------------------
    all_retrieved_chunks = []

    for patient in selected_patients:
        chunks, _ = retrieve_patient_context(message, patient, k=2)
        all_retrieved_chunks.extend(chunks)

    retrieved_context = "\n\n".join(all_retrieved_chunks)

    # -------------------------
    # Build conversation memory
    # -------------------------
    conversation = ""

    for user_msg, bot_msg in history:
        conversation += f"User: {user_msg}\nAssistant: {bot_msg}\n"

    # -------------------------
    # Build prompt
    # -------------------------
    prompt = f"""
You are a clinical reasoning assistant.

Use ONLY the data from the selected patients below.

Structured Clinical Summaries:
{combined_summary}

Relevant Document Excerpts:
{retrieved_context}

Conversation so far:
{conversation}

New Question:
{message}

Do NOT fabricate information.
If data is not available, state clearly.
"""

    # -------------------------
    # Call model
    # -------------------------
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": "You are a safe clinical AI system."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# Build chatUI using Gradio chat interface

In [ ]:
import gradio as gr

patient_list = list(all_documents.keys())

def respond(message, history, selected_patients):
    response = chat_with_memory(message, history, selected_patients)
    history.append((message, response))
    return "", history

with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Multi-Patient Discharge Intelligence Assistant")

    with gr.Tabs():

        # ===============================
        # TAB 1 — Clinical / Comparison Chat
        # ===============================
        with gr.Tab("🏥 Clinical Assistant"):

            patient_dropdown = gr.Dropdown(
                choices=patient_list,
                label="Select Patient(s)",
                multiselect=True
            )

            chatbot = gr.Chatbot(height=500)
            msg = gr.Textbox(label="Ask a clinical or comparison question")

            msg.submit(
                respond,
                inputs=[msg, chatbot, patient_dropdown],
                outputs=[msg, chatbot]
            )

        # ===============================
        # TAB 2 — Project Summary
        # ===============================
        with gr.Tab("📄 Project Summary"):

            gr.Markdown("""
## Project Overview

This project implements a hybrid AI-powered discharge intelligence system.

## Problem Statement

Hospitals face significant challenges in ensuring safe, efficient, and consistent discharge processes.

Common issues include:

- Incomplete discharge documentation
- Poorly structured follow-up instructions
- Inconsistent medication reconciliation
- Lack of visibility into readmission risk indicators
- No easy way to compare discharge quality across patients
- Limited operational insights from unstructured discharge summaries

These gaps can lead to:

- Increased readmission rates
- Delayed discharges
- Medication errors
- Poor patient outcomes
- Higher operational costs

---

## Impact on the Hospital

Without structured discharge intelligence:

- Discharge inefficiencies remain hidden
- Risk factors are not systematically identified
- Process gaps are detected reactively instead of proactively
- Cross-patient trend analysis is difficult
- Clinical and operations teams lack actionable insights

This creates both **clinical risk** and **financial strain**.

---

## Proposed Solution

This project implements a hybrid AI-powered discharge intelligence system combining:

### 1. Structured Clinical Extraction
- Converts unstructured discharge PDFs into standardized JSON
- Extracts demographics, diagnoses, medications, risk indicators, and follow-up data

### 2. Patient-Scoped Retrieval (RAG)
- Embedding-based document indexing
- Safe patient-level retrieval using FAISS
- Guardrails to prevent cross-patient leakage

### 3.Cross-Patient Comparison Mode
- Multi-patient selection
- Structured comparison of LOS, risk indicators, medication changes
- Identification of discharge process gaps
- Operational improvement recommendations

### 4.Interactive Clinical Interface
- Gradio-based UI
- Patient dropdown selection
- Multi-patient comparison capability
- Embedded documentation

---

##  Key Capabilities

- Patient-level clinical reasoning
- Cross-patient discharge comparison
- Risk factor analysis
- Identification of discharge process gaps
- Operational improvement recommendations

---

## Architecture Overview

1. Multi-PDF ingestion
2. Text extraction
3. Chunking with metadata
4. SentenceTransformer embeddings
5. FAISS vector indexing
6. Structured LLM extraction (Groq)
7. Patient-aware RAG reasoning
8. Multi-patient comparison interface

---

## Technology Stack

- Google Colab
- Groq LLM (openai/gpt-oss-120b)
- SentenceTransformers
- FAISS
- Gradio UI

---

## Use Case

Designed as a prototype for:

- Discharge quality monitoring
- Readmission risk insight generation
- Operational discharge optimization
- Clinical documentation intelligence

This system demonstrates how hybrid LLM + RAG architectures can transform unstructured clinical documentation into actionable operational intelligence.
""")

demo.launch(debug=True)

/tmp/ipython-input-993042679.py:27: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500)
/tmp/ipython-input-993042679.py:27: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ddbda1527e48d52fe6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
